# 🎙️ ATC Transcription Variant Generator for RLHF

This notebook generates multiple transcription variants from audio files using different sampling strategies.

**Purpose:** Create genuinely different transcriptions for RLHF (Reinforcement Learning from Human Feedback) training.

---

## 📋 Instructions

1. Click **Runtime** → **Change runtime type** → Select **T4 GPU** (free)
2. Run all cells (Runtime → Run all)
3. Upload your audio file when prompted
4. Get 5 different transcription variants!

---

## 1️⃣ Install Dependencies

In [ ]:
%%capture
!pip install -q transformers torch librosa accelerate

## 2️⃣ Import Libraries

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa
import numpy as np
from google.colab import files
import json
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 3️⃣ Configuration

**Change `MODEL_ID` to use your custom ATC model:**
- Default: `openai/whisper-large-v3`
- Your model: `your-username/whisper-atc-model`

In [ ]:
# ⚙️ Configuration
MODEL_ID = "openai/whisper-large-v3"  # Change this to your ATC model!
LANGUAGE = "en"
NUM_VARIANTS = 5

print(f"📋 Model: {MODEL_ID}")
print(f"🌍 Language: {LANGUAGE}")
print(f"🔢 Variants to generate: {NUM_VARIANTS}")

## 4️⃣ Load Model

This will download the model to Colab's servers (not your computer!)

In [ ]:
print("🔧 Loading model...")
print("   (This will download ~3GB to Colab's servers, not your computer)")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"✅ Model loaded successfully on {device}!")

## 5️⃣ Define Variant Generation Function

In [ ]:
def generate_variants(audio_path, num_variants=5):
    """
    Generate multiple transcription variants using different decoding strategies.
    """
    print(f"\n🎵 Processing audio: {audio_path}")
    
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)
    print(f"   Duration: {len(audio)/16000:.2f}s")
    
    # Preprocess
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)
    
    # Variant configurations with different sampling strategies
    configs = [
        {
            "name": "greedy",
            "description": "Greedy decoding (most confident)",
            "params": {
                "max_new_tokens": 128,
                "do_sample": False,
            }
        },
        {
            "name": "beam_search",
            "description": "Beam search (5 beams)",
            "params": {
                "max_new_tokens": 128,
                "num_beams": 5,
                "do_sample": False,
            }
        },
        {
            "name": "low_temp",
            "description": "Low temperature sampling (T=0.3)",
            "params": {
                "max_new_tokens": 128,
                "do_sample": True,
                "temperature": 0.3,
                "top_k": 50,
                "top_p": 0.95,
            }
        },
        {
            "name": "medium_temp",
            "description": "Medium temperature sampling (T=0.7)",
            "params": {
                "max_new_tokens": 128,
                "do_sample": True,
                "temperature": 0.7,
                "top_k": 50,
                "top_p": 0.9,
            }
        },
        {
            "name": "high_temp",
            "description": "High temperature sampling (T=1.0)",
            "params": {
                "max_new_tokens": 128,
                "do_sample": True,
                "temperature": 1.0,
                "top_k": 40,
                "top_p": 0.85,
            }
        },
    ]
    
    variants = []
    
    print(f"\n📝 Generating {num_variants} variants...\n")
    
    for i, config in enumerate(configs[:num_variants]):
        print(f"  Variant {i+1}/{num_variants}: {config['name']} - {config['description']}")
        
        try:
            # Generate
            predicted_ids = model.generate(
                inputs,
                language=LANGUAGE,
                **config["params"]
            )
            
            # Decode
            transcription = processor.batch_decode(
                predicted_ids,
                skip_special_tokens=True
            )[0].strip()
            
            variant = {
                "rank_position": i + 1,
                "model": config["name"],
                "model_description": config["description"],
                "text": transcription,
                "confidence": 1.0,  # Whisper doesn't provide confidence
                "parameters": config["params"],
            }
            
            variants.append(variant)
            
            # Show preview
            preview = transcription[:60] + "..." if len(transcription) > 60 else transcription
            print(f"     ✓ \"{preview}\"")
            
        except Exception as e:
            print(f"     ✗ Error: {str(e)}")
    
    return variants

## 6️⃣ Upload Audio File

Click the button below to upload your audio file (.mp3, .wav, .flac, etc.)

In [ ]:
print("📤 Upload your audio file:")
uploaded = files.upload()

# Get the uploaded filename
audio_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {audio_filename}")

## 7️⃣ Generate Variants

This will create 5 different transcriptions!

In [ ]:
# Generate variants
variants = generate_variants(audio_filename, NUM_VARIANTS)

print(f"\n✅ Generated {len(variants)} variants successfully!")

## 8️⃣ Display Results

In [ ]:
print("\n" + "="*60)
print("📊 VARIANT RESULTS")
print("="*60)

for i, variant in enumerate(variants):
    print(f"\n📝 Variant {i+1}: {variant['model']}")
    print("-" * 60)
    print(f"Description: {variant['model_description']}")
    print(f"Text: \"{variant['text']}\"")
    print(f"Parameters: {variant['parameters']}")

# Uniqueness analysis
texts = [v['text'].lower().strip() for v in variants]
unique_texts = set(texts)

print("\n" + "="*60)
print("📈 UNIQUENESS ANALYSIS")
print("="*60)
print(f"\nTotal variants: {len(variants)}")
print(f"Unique transcriptions: {len(unique_texts)}")

if len(unique_texts) == 1:
    print("⚠️  All variants produced the same text")
    print("   (This can happen with very clear audio)")
elif len(unique_texts) == len(variants):
    print("✅ All variants are unique!")
else:
    print(f"✅ Generated {len(unique_texts)} different transcriptions")

if len(unique_texts) > 1:
    print("\nUnique transcriptions:")
    for i, text in enumerate(unique_texts, 1):
        print(f"  {i}. \"{text}\"")

## 9️⃣ Export as JSON

Download the results to use in your pipeline

In [ ]:
# Save to JSON
output_filename = "transcription_variants.json"

with open(output_filename, 'w') as f:
    json.dump({
        "audio_file": audio_filename,
        "model": MODEL_ID,
        "variants": variants
    }, f, indent=2)

print(f"💾 Saved to {output_filename}")
print("\n📥 Download the JSON file:")
files.download(output_filename)

---

## 🎯 Next Steps

1. **Download the JSON file** from the cell above
2. **Import to your pipeline** - Use the variants in your RLHF ranking interface
3. **Humans rank them** - Determine which transcription is best
4. **Train your model** - Use the rankings to improve your ATC model

---

## 💡 Tips

- **Use GPU:** Make sure you selected T4 GPU in runtime settings (it's free!)
- **Batch processing:** Upload multiple files and run in a loop
- **Custom model:** Change `MODEL_ID` to your fine-tuned ATC model
- **More variants:** Increase `NUM_VARIANTS` (up to 7 configs available)

---

## 🔗 Integration with Your Pipeline

To use this in your Node.js pipeline, you can:

1. **Manual:** Run Colab → Download JSON → Upload to your database
2. **Semi-auto:** Use Colab API (ngrok tunnel)
3. **Full-auto:** Deploy to HuggingFace Spaces or Google Cloud

Ask me if you need help with integration!
